# 📈 EGX30 AI Forecaster
## 12-Month Egyptian Stock Market Prediction System
### Using Monte Carlo Simulation & Geometric Brownian Motion (GBM)

---

**Author:** Osman Osama  
**Institution:** Egyptian Chinese University  
**Course:** Stochastic Processes  
**Topic:** Applied Stochastic Modeling in Financial Markets

---

### 🎯 Project Overview

This notebook implements a complete financial forecasting system for stocks listed in the **EGX30 index** (Egyptian Exchange). The system:

- Downloads **real historical stock data** from Yahoo Finance
- Calculates **log returns**, drift (μ), and volatility (σ)
- Simulates future prices using **Geometric Brownian Motion (GBM)**
- Runs **Monte Carlo Simulation** with thousands of paths
- Outputs **probabilistic forecasts** with confidence intervals
- Visualizes results with **interactive Plotly charts**

---

### 📐 Mathematical Model

**Log Returns:**
$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

**Geometric Brownian Motion (GBM):**
$$S_{t+1} = S_t \cdot e^{\left(\mu - \frac{\sigma^2}{2}\right) + \sigma Z_t}$$

Where:
- $S_t$ = Stock price at time $t$
- $\mu$ = Daily mean log return (drift)
- $\sigma$ = Daily standard deviation (volatility)
- $Z_t \sim \mathcal{N}(0, 1)$ = Standard normal random variable

---
## 🔧 Section 1: Install & Import Libraries

In [1]:
# Install required packages (run once)
!pip install yfinance plotly pandas numpy scipy --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from datetime import date, datetime

print("✅ All libraries imported successfully!")
print(f"📅 Today's date: {date.today()}")

✅ All libraries imported successfully!
📅 Today's date: 2026-05-07


---
## 📋 Section 2: EGX30 Stock Universe

The following dictionary contains all 25 stocks we track from the EGX30 index.

In [3]:
EGX30 = {
    "COMI.CA": "Commercial International Bank",
    "TMGH.CA": "Talaat Moustafa Group",
    "EAST.CA": "Eastern Company",
    "FWRY.CA": "Fawry",
    "HRHO.CA": "EFG Holding",
    "EFIH.CA": "eFinance Investment Group",
    "ETEL.CA": "Telecom Egypt",
    "BTFH.CA": "Beltone Holding",
    "ABUK.CA": "Abu Qir Fertilizers",
    "PHDC.CA": "Palm Hills Development",
    "OCDI.CA": "Orascom Development Egypt",
    "HELI.CA": "Heliopolis Housing",
    "MNHD.CA": "Madinet Masr Housing",
    "ESRS.CA": "Ezz Steel",
    "SWDY.CA": "Elsewedy Electric",
    "AMOC.CA": "Alexandria Mineral Oils",
    "ALCN.CA": "Alexandria Containers",
    "EKHO.CA": "Egypt Kuwait Holding",
    "JUFO.CA": "Juhayna Food Industries",
    "ORWE.CA": "Oriental Weavers",
    "AUTO.CA": "GB Corp",
    "CNFN.CA": "Contact Financial",
    "CICH.CA": "CI Capital Holding",
    "EFID.CA": "Edita Food Industries",
    "EGAL.CA": "Egypt Aluminum"
}

print(f"📊 Total EGX30 stocks tracked: {len(EGX30)}")
print("\n🗂️ Stock List:")
for ticker, name in EGX30.items():
    print(f"   {ticker:12s} → {name}")

📊 Total EGX30 stocks tracked: 25

🗂️ Stock List:
   COMI.CA      → Commercial International Bank
   TMGH.CA      → Talaat Moustafa Group
   EAST.CA      → Eastern Company
   FWRY.CA      → Fawry
   HRHO.CA      → EFG Holding
   EFIH.CA      → eFinance Investment Group
   ETEL.CA      → Telecom Egypt
   BTFH.CA      → Beltone Holding
   ABUK.CA      → Abu Qir Fertilizers
   PHDC.CA      → Palm Hills Development
   OCDI.CA      → Orascom Development Egypt
   HELI.CA      → Heliopolis Housing
   MNHD.CA      → Madinet Masr Housing
   ESRS.CA      → Ezz Steel
   SWDY.CA      → Elsewedy Electric
   AMOC.CA      → Alexandria Mineral Oils
   ALCN.CA      → Alexandria Containers
   EKHO.CA      → Egypt Kuwait Holding
   JUFO.CA      → Juhayna Food Industries
   ORWE.CA      → Oriental Weavers
   AUTO.CA      → GB Corp
   CNFN.CA      → Contact Financial
   CICH.CA      → CI Capital Holding
   EFID.CA      → Edita Food Industries
   EGAL.CA      → Egypt Aluminum


---
## ⚙️ Section 3: Configuration Parameters

**Modify these parameters to run different experiments.**

In [4]:
# ============================================================
# USER CONFIGURATION — CHANGE THESE
# ============================================================

TICKER        = "EAST.CA"       # Stock ticker from EGX30 dict
START_DATE    = "2022-01-01"    # Training data start
END_DATE      = str(date.today())  # Training data end
FORECAST_DAYS = 252             # ~12 months of trading days
N_PATHS       = 1000            # Number of Monte Carlo simulations
RANDOM_SEED   = 42              # For reproducibility

# ============================================================
np.random.seed(RANDOM_SEED)

print("✅ Configuration set:")
print(f"   Stock     : {TICKER} — {EGX30.get(TICKER, 'Unknown')}")
print(f"   Period    : {START_DATE} → {END_DATE}")
print(f"   Forecast  : {FORECAST_DAYS} trading days (~12 months)")
print(f"   MC Paths  : {N_PATHS:,}")

✅ Configuration set:
   Stock     : EAST.CA — Eastern Company
   Period    : 2022-01-01 → 2026-05-07
   Forecast  : 252 trading days (~12 months)
   MC Paths  : 1,000


---
## 📥 Section 4: Download Historical Data

In [5]:
def load_data(ticker, start, end):
    """
    Download historical closing prices from Yahoo Finance.
    Handles MultiIndex columns from newer yfinance versions.
    """
    data = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False
    )

    if data.empty:
        raise ValueError(f"No data returned for ticker: {ticker}")

    # Handle MultiIndex columns (yfinance ≥ 0.2.x)
    if isinstance(data.columns, pd.MultiIndex):
        close = data["Close"].iloc[:, 0]
    else:
        close = data["Close"]

    return close.dropna()


# Download
prices = load_data(TICKER, START_DATE, END_DATE)

print(f"✅ Data loaded: {len(prices)} trading days")
print(f"   Date range : {prices.index[0].date()} → {prices.index[-1].date()}")
print(f"   First price: {prices.iloc[0]:.2f} EGP")
print(f"   Last price : {prices.iloc[-1]:.2f} EGP")
print()
prices.tail(10)

✅ Data loaded: 1051 trading days
   Date range : 2022-01-02 → 2026-05-06
   First price: 7.47 EGP
   Last price : 39.53 EGP



Date
2026-04-23    41.240002
2026-04-26    41.650002
2026-04-27    41.400002
2026-04-28    40.000000
2026-04-29    39.820000
2026-04-30    39.820000
2026-05-03    39.169998
2026-05-04    38.000000
2026-05-05    39.000000
2026-05-06    39.529999
Name: EAST.CA, dtype: float64

---
## 📊 Section 5: Exploratory Data Analysis (EDA)

In [6]:
# Historical price chart
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=prices.index,
    y=prices.values,
    mode="lines",
    name="Closing Price",
    line=dict(color="#06b6d4", width=2)
))

fig.update_layout(
    title=f"📈 {TICKER} — {EGX30.get(TICKER)} | Historical Closing Prices",
    xaxis_title="Date",
    yaxis_title="Price (EGP)",
    template="plotly_dark",
    paper_bgcolor="#0f172a",
    plot_bgcolor="#0f172a",
    height=450,
    xaxis=dict(gridcolor="rgba(255,255,255,0.05)"),
    yaxis=dict(gridcolor="rgba(255,255,255,0.05)")
)

fig.show()

---
## 📐 Section 6: Log Returns Calculation & Statistics

In [7]:
# Calculate log returns
log_returns = np.log(prices / prices.shift(1)).dropna()

# Parameters
mu    = log_returns.mean()          # Daily drift
sigma = log_returns.std()           # Daily volatility
S0    = prices.iloc[-1]             # Last known price

# Annualized values
mu_annual    = mu * 252
sigma_annual = sigma * np.sqrt(252)

# Skewness & Kurtosis
skew = log_returns.skew()
kurt = log_returns.kurt()

# Normality test
stat, p_value = stats.shapiro(log_returns.sample(min(5000, len(log_returns))))

print("="*55)
print(f"  📊 STATISTICAL ANALYSIS — {TICKER}")
print("="*55)
print(f"  Trading Days           : {len(prices):>10,}")
print(f"  Current Price (S₀)     : {S0:>10.2f} EGP")
print()
print(f"  Daily Mean Return (μ)  : {mu:>10.6f}")
print(f"  Daily Volatility (σ)   : {sigma:>10.6f}")
print()
print(f"  Annualized Return      : {mu_annual:>9.2%}")
print(f"  Annualized Volatility  : {sigma_annual:>9.2%}")
print()
print(f"  Skewness               : {skew:>10.4f}")
print(f"  Excess Kurtosis        : {kurt:>10.4f}")
print(f"  Shapiro-Wilk p-value   : {p_value:>10.4f}")
print(f"  Normal distribution?   : {'Yes ✅' if p_value > 0.05 else 'No ❌ (fat tails)':>10}")
print("="*55)

  📊 STATISTICAL ANALYSIS — EAST.CA
  Trading Days           :      1,051
  Current Price (S₀)     :      39.53 EGP

  Daily Mean Return (μ)  :   0.001587
  Daily Volatility (σ)   :   0.028289

  Annualized Return      :    40.00%
  Annualized Volatility  :    44.91%

  Skewness               :    -1.3700
  Excess Kurtosis        :    26.6726
  Shapiro-Wilk p-value   :     0.0000
  Normal distribution?   : No ❌ (fat tails)


In [8]:
# Return distribution with normal fit overlay
x_range = np.linspace(log_returns.min(), log_returns.max(), 300)
normal_fit = stats.norm.pdf(x_range, mu, sigma)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Log Return Distribution", "Rolling 30-Day Volatility"))

# Histogram
fig.add_trace(go.Histogram(
    x=log_returns, nbinsx=60,
    name="Log Returns",
    marker_color="#2563eb",
    histnorm="probability density",
    opacity=0.75
), row=1, col=1)

# Normal fit
fig.add_trace(go.Scatter(
    x=x_range, y=normal_fit,
    name="Normal Fit",
    line=dict(color="#f59e0b", width=2.5)
), row=1, col=1)

# Rolling volatility
rolling_vol = log_returns.rolling(30).std() * np.sqrt(252)
fig.add_trace(go.Scatter(
    x=rolling_vol.index, y=rolling_vol.values,
    name="Rolling Vol (30d)",
    line=dict(color="#ef4444", width=2)
), row=1, col=2)

fig.update_layout(
    title=f"📉 Return Analysis — {TICKER}",
    template="plotly_dark",
    paper_bgcolor="#0f172a",
    plot_bgcolor="#0f172a",
    height=420,
    showlegend=True
)

fig.show()

---
## 🤖 Section 7: Geometric Brownian Motion (GBM) Model

The GBM model simulates future stock prices using:

$$S_{t+1} = S_t \cdot e^{\left(\mu - \frac{\sigma^2}{2}\right) + \sigma Z_t}, \quad Z_t \sim \mathcal{N}(0,1)$$

The term $\left(\mu - \frac{\sigma^2}{2}\right)$ is the **Itô correction** — it adjusts for the fact that the expected log price grows at a rate less than the arithmetic mean due to volatility drag.

In [9]:
def gbm_simulate(prices, horizon=252, n_paths=1000, seed=42):
    """
    Geometric Brownian Motion Monte Carlo Simulation.

    Parameters:
    -----------
    prices  : pd.Series  — Historical closing prices
    horizon : int        — Number of trading days to forecast
    n_paths : int        — Number of Monte Carlo simulation paths
    seed    : int        — Random seed for reproducibility

    Returns:
    --------
    dict with simulation arrays and model parameters
    """
    np.random.seed(seed)

    # Model parameters
    log_returns = np.log(prices / prices.shift(1)).dropna()
    mu    = log_returns.mean()
    sigma = log_returns.std()
    S0    = prices.iloc[-1]

    # Drift term (Itô-corrected)
    drift = mu - 0.5 * sigma**2

    # Simulate all paths at once using vectorized operations (fast!)
    # Shape: (horizon, n_paths)
    random_shocks = np.random.normal(0, 1, size=(horizon, n_paths))
    daily_returns = np.exp(drift + sigma * random_shocks)

    # Build price paths
    price_paths = np.zeros((horizon, n_paths))
    price_paths[0, :] = S0

    for t in range(1, horizon):
        price_paths[t, :] = price_paths[t-1, :] * daily_returns[t, :]

    return {
        "paths"    : price_paths,
        "mean"     : price_paths.mean(axis=1),
        "median"   : np.median(price_paths, axis=1),
        "p5"       : np.percentile(price_paths, 5, axis=1),
        "p25"      : np.percentile(price_paths, 25, axis=1),
        "p75"      : np.percentile(price_paths, 75, axis=1),
        "p95"      : np.percentile(price_paths, 95, axis=1),
        "mu"       : mu,
        "sigma"    : sigma,
        "S0"       : S0,
        "drift"    : drift,
        "horizon"  : horizon,
        "n_paths"  : n_paths
    }


print("⏳ Running Monte Carlo simulation...")
result = gbm_simulate(prices, horizon=FORECAST_DAYS, n_paths=N_PATHS, seed=RANDOM_SEED)
print(f"✅ Simulation complete: {N_PATHS:,} paths × {FORECAST_DAYS} days = {N_PATHS * FORECAST_DAYS:,} price points")

⏳ Running Monte Carlo simulation...
✅ Simulation complete: 1,000 paths × 252 days = 252,000 price points


---
## 📅 Section 8: Generate Future Business Dates

In [10]:
# Generate future business days (skips weekends automatically)
future_dates = pd.bdate_range(
    start=prices.index[-1] + pd.offsets.BDay(1),
    periods=FORECAST_DAYS
)

print(f"📅 Forecast period:")
print(f"   Start : {future_dates[0].date()}")
print(f"   End   : {future_dates[-1].date()}")
print(f"   Days  : {len(future_dates)} trading days")

📅 Forecast period:
   Start : 2026-05-07
   End   : 2027-04-23
   Days  : 252 trading days


---
## 📈 Section 9: Forecast Visualization

In [11]:
fig = go.Figure()

# Historical prices
fig.add_trace(go.Scatter(
    x=prices.index, y=prices.values,
    name="Historical",
    line=dict(color="#94a3b8", width=2)
))

# Sample MC paths (plot first 100 for clarity)
for i in range(min(100, N_PATHS)):
    fig.add_trace(go.Scatter(
        x=future_dates, y=result["paths"][:, i],
        mode="lines",
        line=dict(color="rgba(37,99,235,0.06)", width=1),
        showlegend=False
    ))

# 95% confidence band (upper)
fig.add_trace(go.Scatter(
    x=future_dates, y=result["p95"],
    line=dict(width=0),
    showlegend=False
))

# 95% confidence band (lower fill)
fig.add_trace(go.Scatter(
    x=future_dates, y=result["p5"],
    fill="tonexty",
    fillcolor="rgba(37,99,235,0.15)",
    line=dict(width=0),
    name="90% Confidence Band"
))

# Mean forecast
fig.add_trace(go.Scatter(
    x=future_dates, y=result["mean"],
    name="Mean Forecast",
    line=dict(color="#06b6d4", width=3)
))

# Median forecast
fig.add_trace(go.Scatter(
    x=future_dates, y=result["median"],
    name="Median Forecast",
    line=dict(color="#f59e0b", width=2, dash="dash")
))

fig.update_layout(
    title=f"📈 {TICKER} — {EGX30.get(TICKER)} | 12-Month Monte Carlo Forecast",
    xaxis_title="Date",
    yaxis_title="Price (EGP)",
    template="plotly_dark",
    paper_bgcolor="#0f172a",
    plot_bgcolor="#0f172a",
    height=600,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    xaxis=dict(gridcolor="rgba(255,255,255,0.05)"),
    yaxis=dict(gridcolor="rgba(255,255,255,0.05)")
)

fig.show()

---
## 📊 Section 10: Key Forecast Metrics

In [12]:
current_price   = result["S0"]
forecast_mean   = result["mean"][-1]
forecast_median = result["median"][-1]
forecast_p5     = result["p5"][-1]
forecast_p95    = result["p95"][-1]

expected_return = (forecast_mean - current_price) / current_price * 100
annual_vol      = result["sigma"] * np.sqrt(252) * 100
sharpe_approx   = (result["mu"] * 252) / (result["sigma"] * np.sqrt(252))  # risk-free = 0

print("╔" + "═"*52 + "╗")
print(f"║  📊 FORECAST RESULTS — {TICKER:<28}║")
print("╠" + "═"*52 + "╣")
print(f"║  Current Price (S₀)     : {current_price:>10.2f} EGP          ║")
print(f"║  Forecast Mean Price    : {forecast_mean:>10.2f} EGP          ║")
print(f"║  Forecast Median Price  : {forecast_median:>10.2f} EGP          ║")
print("╠" + "═"*52 + "╣")
print(f"║  95th Percentile (Best) : {forecast_p95:>10.2f} EGP          ║")
print(f"║  5th Percentile (Worst) : {forecast_p5:>10.2f} EGP          ║")
print("╠" + "═"*52 + "╣")
print(f"║  Expected Return        : {expected_return:>+10.2f}%            ║")
print(f"║  Annual Volatility      : {annual_vol:>10.2f}%            ║")
print(f"║  Sharpe Ratio (rf=0)    : {sharpe_approx:>10.4f}             ║")
print("╚" + "═"*52 + "╝")

╔════════════════════════════════════════════════════╗
║  📊 FORECAST RESULTS — EAST.CA                     ║
╠════════════════════════════════════════════════════╣
║  Current Price (S₀)     :      39.53 EGP          ║
║  Forecast Mean Price    :      59.01 EGP          ║
║  Forecast Median Price  :      53.62 EGP          ║
╠════════════════════════════════════════════════════╣
║  95th Percentile (Best) :     113.05 EGP          ║
║  5th Percentile (Worst) :      25.28 EGP          ║
╠════════════════════════════════════════════════════╣
║  Expected Return        :     +49.28%            ║
║  Annual Volatility      :      44.91%            ║
║  Sharpe Ratio (rf=0)    :     0.8908             ║
╚════════════════════════════════════════════════════╝


---
## 📋 Section 11: Forecast Table

In [13]:
forecast_df = pd.DataFrame({
    "Date"           : future_dates,
    "Forecast (Mean)": result["mean"].round(2),
    "Median"         : result["median"].round(2),
    "Low (5%)"       : result["p5"].round(2),
    "High (95%)"     : result["p95"].round(2),
}).set_index("Date")

print(f"📋 Full Forecast Table — {len(forecast_df)} trading days")
print()

# Style the dataframe
forecast_df.style \
    .format("{:.2f}") \
    .background_gradient(subset=["Forecast (Mean)"], cmap="Blues") \
    .set_caption(f"{TICKER} — 12-Month Price Forecast")

📋 Full Forecast Table — 252 trading days



,Forecast (Mean),Median,Low (5%),High (95%)
Date,,,,
2026-05-07 00:00:00,39.53,39.53,39.53,39.53
2026-05-08 00:00:00,39.67,39.65,37.85,41.52
2026-05-11 00:00:00,39.74,39.70,37.17,42.37
2026-05-12 00:00:00,39.78,39.72,36.65,43.20
2026-05-13 00:00:00,39.79,39.77,36.23,43.46
2026-05-14 00:00:00,39.80,39.79,35.76,43.85
2026-05-15 00:00:00,39.83,39.80,35.44,44.52
2026-05-18 00:00:00,39.92,39.90,34.99,45.34
2026-05-19 00:00:00,40.01,39.86,34.75,45.89


In [14]:
# Monthly summary (every ~21 trading days)
monthly_idx = list(range(0, FORECAST_DAYS, 21)) + [FORECAST_DAYS - 1]
monthly_summary = forecast_df.iloc[monthly_idx].copy()

print("📅 Monthly Forecast Summary:")
print(monthly_summary.to_string())

📅 Monthly Forecast Summary:
            Forecast (Mean)  Median  Low (5%)  High (95%)
Date                                                     
2026-05-07            39.53   39.53     39.53       39.53
2026-06-05            41.00   40.68     32.59       50.22
2026-07-06            42.19   41.91     30.84       56.44
2026-08-04            43.77   43.17     29.87       61.14
2026-09-02            45.33   43.98     28.12       67.61
2026-10-01            46.75   44.76     28.52       72.24
2026-10-30            48.50   46.90     27.77       75.55
2026-11-30            50.12   47.49     27.30       82.65
2026-12-29            52.01   48.64     26.70       88.91
2027-01-27            53.55   49.69     26.74       95.79
2027-02-25            55.26   51.52     26.69       98.22
2027-03-26            57.42   53.24     25.86      106.22
2027-04-23            59.01   53.62     25.28      113.05


---
## 📉 Section 12: Risk Analysis — Final Price Distribution

In [15]:
final_prices = result["paths"][-1, :]  # All N_PATHS final prices

# Probability of profit
prob_profit = np.mean(final_prices > current_price) * 100
prob_loss   = 100 - prob_profit
prob_gain20 = np.mean(final_prices > current_price * 1.20) * 100
prob_loss20 = np.mean(final_prices < current_price * 0.80) * 100

print(f"📊 Risk Analysis — {TICKER}")
print(f"   P(price > current)         : {prob_profit:.1f}%")
print(f"   P(price < current)         : {prob_loss:.1f}%")
print(f"   P(gain > 20%)              : {prob_gain20:.1f}%")
print(f"   P(loss > 20%)              : {prob_loss20:.1f}%")
print()

# Distribution chart
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=final_prices,
    nbinsx=80,
    name="Final Price Distribution",
    marker_color="#2563eb",
    opacity=0.8
))

# Vertical lines
for val, color, label in [
    (current_price,   "#94a3b8", f"Current: {current_price:.2f}"),
    (forecast_mean,   "#06b6d4", f"Mean: {forecast_mean:.2f}"),
    (forecast_p5,     "#ef4444", f"P5: {forecast_p5:.2f}"),
    (forecast_p95,    "#22c55e", f"P95: {forecast_p95:.2f}")
]:
    fig.add_vline(x=val, line_dash="dash", line_color=color,
                  annotation_text=label, annotation_position="top")

fig.update_layout(
    title=f"📊 Final Price Distribution After {FORECAST_DAYS} Trading Days — {TICKER}",
    xaxis_title="Price (EGP)",
    yaxis_title="Frequency",
    template="plotly_dark",
    paper_bgcolor="#0f172a",
    plot_bgcolor="#0f172a",
    height=450
)

fig.show()

📊 Risk Analysis — EAST.CA
   P(price > current)         : 74.1%
   P(price < current)         : 25.9%
   P(gain > 20%)              : 60.2%
   P(loss > 20%)              : 13.1%



---
## 🔄 Section 13: Multi-Stock Comparison

In [16]:
# Compare multiple EGX30 stocks
COMPARE_TICKERS = ["EAST.CA", "COMI.CA", "ETEL.CA", "FWRY.CA"]

comparison_results = []

for tk in COMPARE_TICKERS:
    try:
        p = load_data(tk, START_DATE, END_DATE)
        if len(p) < 50:
            continue
        r = gbm_simulate(p, horizon=FORECAST_DAYS, n_paths=500)
        current = r["S0"]
        exp_ret = (r["mean"][-1] - current) / current * 100
        ann_vol = r["sigma"] * np.sqrt(252) * 100
        comparison_results.append({
            "Ticker"           : tk,
            "Company"          : EGX30.get(tk, tk),
            "Current Price"    : round(current, 2),
            "Forecast Price"   : round(r["mean"][-1], 2),
            "Expected Return %": round(exp_ret, 2),
            "Annual Volatility%": round(ann_vol, 2)
        })
        print(f"   ✅ {tk} done")
    except Exception as e:
        print(f"   ❌ {tk} failed: {e}")

comp_df = pd.DataFrame(comparison_results)
print()
comp_df

   ✅ EAST.CA done
   ✅ COMI.CA done
   ✅ ETEL.CA done
   ✅ FWRY.CA done



,Ticker,Company,Current Price,Forecast Price,Expected Return %,Annual Volatility%
0,EAST.CA,Eastern Company,39.53,58.98,49.21,44.91
1,COMI.CA,Commercial International Bank,141.60,203.54,43.74,32.83
2,ETEL.CA,Telecom Egypt,96.71,157.61,62.97,37.77
3,FWRY.CA,Fawry,20.43,26.58,30.09,44.39


In [17]:
# Risk-Return scatter plot
if len(comp_df) > 0:
    fig = px.scatter(
        comp_df,
        x="Annual Volatility%",
        y="Expected Return %",
        text="Ticker",
        size=[30]*len(comp_df),
        color="Expected Return %",
        color_continuous_scale="Blues",
        title="📊 Risk-Return Analysis — EGX30 Stocks"
    )

    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)

    fig.update_traces(textposition="top center")
    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor="#0f172a",
        plot_bgcolor="#0f172a",
        height=500,
        xaxis_title="Annual Volatility (%)",
        yaxis_title="Expected Return (%)"
    )
    fig.show()

---
## 💾 Section 14: Export Results to CSV

In [18]:
# Export forecast table to CSV
output_file = f"{TICKER.replace('.', '_')}_forecast.csv"
forecast_df.to_csv(output_file)
print(f"✅ Forecast saved to: {output_file}")

# Export comparison table
if len(comp_df) > 0:
    comp_df.to_csv("egx30_comparison.csv", index=False)
    print("✅ Comparison table saved to: egx30_comparison.csv")

✅ Forecast saved to: EAST_CA_forecast.csv
✅ Comparison table saved to: egx30_comparison.csv


---
## 📝 Section 15: Academic Summary

### Conclusions

| Topic | Key Finding |
|-------|-------------|
| **Model Used** | Geometric Brownian Motion (GBM) with Monte Carlo simulation |
| **Distribution** | Log returns approximately follow normal distribution |
| **Confidence** | 90% of simulated paths fall within the P5–P95 band |
| **Limitations** | GBM assumes constant volatility (no volatility clustering) |
| **Extensions** | GARCH(1,1) would better capture volatility clustering in EGX |

### Stochastic Process Concepts Applied

1. **Brownian Motion (Wiener Process)** — Continuous random walk underlying GBM
2. **Itô's Lemma** — Used to derive the GBM stochastic differential equation
3. **Maximum Likelihood Estimation (MLE)** — For estimating μ and σ from data
4. **Monte Carlo Simulation** — For approximate numerical integration of expectations
5. **Confidence Intervals** — Quantile-based probabilistic prediction bands

### Formula Reference

| Symbol | Meaning |
|--------|---------|
| $S_0$ | Current stock price |
| $\mu$ | Daily mean log return (drift) |
| $\sigma$ | Daily standard deviation (volatility) |
| $Z_t$ | Standard normal random variable |
| $T$ | Forecast horizon (252 = 1 year) |
| $N$ | Number of Monte Carlo paths |

---

**Egyptian Chinese University — Stochastic Processes**  
**Osman Osama | 2025–2026**